# Breast Cancer Classification using K-Nearest Neighbours (KNN)

In this notebook, we use the **Breast Cancer dataset** from sklearn to classify tumours as either:
- **0** = Malignant (cancerous)
- **1** = Benign (non-cancerous)

We follow the standard Machine Learning pipeline:
1. Load the dataset
2. Split into train and test sets
3. Train the KNN model
4. Predict on the test set
5. Evaluate using a confusion matrix, accuracy, precision, and recall

## Step 1 — Import Libraries

We import the core libraries needed:
- `numpy` for numerical operations
- `sklearn` for datasets, models, and metrics
- `matplotlib` for plotting

In [25]:
import numpy as np
import sklearn
from sklearn import datasets
import matplotlib.pyplot as plt

## Step 2 — Load the Dataset

We load the **Breast Cancer dataset** which is built into sklearn.

- `bb.data` → the 30 feature measurements (e.g. tumour size, texture, smoothness)
- `bb.target` → the class labels (0 = malignant, 1 = benign)

We assign them to `X` (features) and `y` (labels) — standard ML convention.

In [26]:
# Load the breast cancer dataset from sklearn
bb = datasets.load_breast_cancer()

# X = features (30 measurements per tumour sample)
X = bb.data

# y = target labels (0 = malignant, 1 = benign)
y = bb.target

## Step 3 — Split into Train and Test Sets

We split the data into:
- **80% training** → used to teach the model
- **20% testing** → used to evaluate the model on unseen data

This is done **before** fitting the model to avoid **data leakage** — the model should never see the test data during training.

The correct order of variables is always:
```
X_train, X_test, y_train, y_test
```

In [27]:
from sklearn.model_selection import train_test_split

# Split: 80% train, 20% test
# X_train, X_test = feature splits
# ytrain, ytest   = label splits
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2)

## Step 4 — Train the KNN Model and Predict

**K-Nearest Neighbours (KNN)** classifies a new data point by looking at the `k` closest points in the training set and taking a majority vote.

- `n_neighbors=5` → look at the 5 nearest neighbours (default and common starting point)
- `knn.fit(Xtrain, ytrain)` → the model learns from the training data
- `knn.predict(Xtest)` → the model predicts labels for the unseen test features

Note: we only pass `Xtest` to predict — passing `ytest` would be cheating as it contains the real answers.

In [28]:
from sklearn.neighbors import KNeighborsClassifier

# Create the KNN model with k=5 neighbours
knn = KNeighborsClassifier(n_neighbors=5)

# Train the model using only training data
knn.fit(Xtrain, ytrain)

# Predict labels for the test set (model has never seen Xtest before)
ypredtest = knn.predict(Xtest)

## Step 5 — Confusion Matrix

A **confusion matrix** shows how many predictions were correct and incorrect for each class.

For a binary classification (2 classes):

|  | Predicted 0 | Predicted 1 |
|---|---|---|
| **Actual 0** | True Negative (TN) | False Positive (FP) |
| **Actual 1** | False Negative (FN) | True Positive (TP) |

- **Diagonal values** = correct predictions
- **Off-diagonal values** = mistakes

The order of arguments is always: `confusion_matrix(actual, predicted)`

In [29]:
from sklearn.metrics import confusion_matrix

# Build the confusion matrix — actual labels first, predicted second
cm = confusion_matrix(ytest, ypredtest)

# Visualise the confusion matrix as a colour map
plt.matshow(cm)

# Print the raw numbers
print(cm)

## Step 6 — Custom Accuracy Function

**Accuracy** measures the proportion of correct predictions out of all predictions.

$$Accuracy = \frac{\text{Total Correct Predictions}}{\text{Total Predictions}}$$

Using the confusion matrix:
- `np.diag(cm).sum()` → sum of diagonal = all correct predictions
- `cm.sum()` → sum of entire matrix = total samples

We use the confusion matrix here to practice reading diagonal values — it gives the same result as a direct comparison.

In [39]:
def accuracy(ytest, ypredtest):
    # Build confusion matrix
    cm = confusion_matrix(ytest, ypredtest)
    
    # Diagonal = correct predictions, cm.sum() = total samples
    accuracy = np.diag(cm).sum() / cm.sum()
    
    return accuracy

## Step 7 — Custom Precision Function

**Precision** measures — of all the times the model predicted a class, how often was it right?

$$Precision_i = \frac{TP}{TP + FP}$$

Precision is calculated **per column** of the confusion matrix:
- `cm.T` → transpose the matrix so columns become rows (easier to loop)
- `col[i]` → diagonal element = True Positives for class i
- `col.sum()` → entire column = TP + FP

**Macro average** = calculate precision for each class separately, then take the mean — treats all classes equally.

In [44]:
def precesion(ytest, ypredtest):
    prec = []  # list to store precision for each class
    
    # Build confusion matrix
    cm = confusion_matrix(ytest, ypredtest)
    
    # Loop over columns (transpose so columns become rows)
    # i = class index, col = that class's column
    for i, col in enumerate(cm.T):
        # col[i] = True Positives, col.sum() = TP + FP
        preci = col[i] / col.sum()
        prec.append(preci)
    
    # Return macro average (mean across all classes)
    return np.mean(prec)

## Step 8 — Custom Recall Function

**Recall** measures — of all the actual instances of a class, how many did the model correctly identify?

$$Recall_i = \frac{TP}{TP + FN}$$

Recall is calculated **per row** of the confusion matrix:
- `row[i]` → diagonal element = True Positives for class i
- `row.sum()` → entire row = TP + FN

**Macro average** = mean of recall across all classes.

In [50]:
def recall(ytest, ypredtest):
    recall = []  # list to store recall for each class
    
    # Build confusion matrix
    cm = confusion_matrix(ytest, ypredtest)
    
    # Loop over rows — each row represents one actual class
    # i = class index, row = that class's row
    for i, row in enumerate(cm):
        # row[i] = True Positives, row.sum() = TP + FN
        reci = row[i] / row.sum()
        recall.append(reci)
    
    # Return macro average (mean across all classes)
    return np.mean(recall)

## Step 9 — Verify Against Sklearn

We verify our custom functions match sklearn's built-in functions.

- `average='macro'` → tells sklearn to use macro averaging (same as our functions)
- All comparisons should return `True` if our implementations are correct

Note: We use `==` here because our functions and sklearn use the same arithmetic — results match exactly.

In [51]:
from sklearn.metrics import precision_score, recall_score, accuracy_score

# Verify accuracy — should return True
accuracy(ytest, ypredtest) == accuracy_score(ytest, ypredtest)

In [52]:
# Verify precision — should return True
precesion(ytest, ypredtest) == precision_score(ytest, ypredtest, average='macro')

In [53]:
# Verify recall — should return True
recall(ytest, ypredtest) == recall_score(ytest, ypredtest, average='macro')